In [3]:
# Install required packages (run once) - PyTorch 2.9.1 + CUDA 12.6
!pip install --quiet torch==2.9.1+cu126 torchvision==0.24.1+cu126 torchaudio==2.9.1+cu126 --extra-index-url https://download.pytorch.org/whl/cu126
!pip install --quiet timm albumentations==1.3.0 "datasets[vision]==2.18.0" torchmetrics scikit-learn opencv-python-headless matplotlib seaborn pandas tqdm

In [4]:
# Check packages and CUDA version
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.9.1+cu126
True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
# Logging in HuggingFace
from huggingface_hub import login
login()


In [5]:
# Verify login
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '696159abaa6b67e1b6bf7790', 'name': 'infinityxr9', 'fullname': 'Aryan Sisodiya', 'email': 'aryansisodiya091107@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1769904000, 'isPro': False, 'avatarUrl': '/avatars/3f0547d01959ef03971434d884a9641f.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ps1', 'role': 'read', 'createdAt': '2026-01-09T19:51:02.375Z'}}}


# Galaxy Morphology Classifier — SPACECODE 2026 (Problem-1)

## Overview
This notebook implements a complete **reproducible research pipeline** for classifying galaxy morphologies using deep learning. The dataset contains 256×256 RGB images of galaxies across 10 morphological classes from the HuggingFace repository `Xanadu00/autotrain-data-galaxy_classification`.

## Pipeline Structure
1. **Data Loading & Preprocessing** - Load dataset, analyze class distribution, compute class weights for imbalanced data
2. **Augmentation** - Apply Albumentations transforms (rotation, flipping, cropping, brightness, noise)
3. **Model Architecture** - Pretrained ConvNeXt-Tiny with custom classification head
4. **Training** - Mixed precision training with weighted loss, cosine learning rate scheduling, and early stopping
5. **Evaluation** - Confusion matrix, per-class metrics (precision, recall, F1), ROC-AUC
6. **Robustness Testing** - Evaluate model performance under realistic image corruptions (blur, noise, compression)

## Requirements
- GPU with CUDA support (recommended)
- Python 3.8+
- Internet connection for downloading dataset and pretrained weights

**Note:** All random seeds are fixed for reproducibility.

In [6]:
# Imports and deterministic setup
import os, random, time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.utils.class_weight import compute_class_weight
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Device: cuda
PyTorch version: 2.9.1+cu126
CUDA available: True


In [ ]:
# 1) Data: load HF dataset, compute class distribution and class weights
DATASET_ID = 'Xanadu00/autotrain-data-galaxy_classification'
ds = load_dataset(DATASET_ID)
# Inspect splits
print(ds)
# Convert to train/validation if present; otherwise split train
if 'train' in ds and 'validation' in ds:
    ds_train = ds['train']
    ds_val = ds['validation']
else:
    ds = ds['train'].train_test_split(test_size=0.2, seed=SEED)
    ds_train = ds['train']
    ds_val = ds['test']
# Show class distribution
train_labels = np.array(ds_train['label'])
unique, counts = np.unique(train_labels, return_counts=True)
print('Train class distribution:')
for u, c in zip(unique, counts):
    print(f'  Class {u}: {c}')
# Compute class weights (for weighted CE)
classes = np.unique(train_labels)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print('Class weights:', class_weights)

Resolving data files:   0%|          | 0/28382 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3551 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3548 [00:00<?, ?it/s]

## 1. Data Pipeline

### Dataset
Loading the galaxy classification dataset from HuggingFace. The dataset contains astronomical images of galaxies labeled by morphological type (spiral, elliptical, irregular, etc.).

### Class Imbalance Handling
Computing class weights using sklearn's `compute_class_weight` with the 'balanced' strategy. This ensures that minority classes receive higher weights during training, preventing the model from being biased towards majority classes.

### Augmentation Strategy
**Training augmentations:**
- **RandomRotate90**: Galaxies have rotational symmetry
- **Flip**: Horizontal/vertical flipping preserves galaxy properties
- **RandomResizedCrop**: Slight scale variations simulate different observation distances
- **RandomBrightnessContrast**: Mimics varying telescope exposure and atmospheric conditions
- **GaussNoise**: Models sensor noise in astronomical imaging

**Validation/Test transforms:**
- Simple resize and normalization (no augmentation to ensure consistent evaluation)

In [ ]:
# Albumentations transforms
IMG_SIZE = 256
train_transforms = A.Compose([
    A.RandomRotate90(),
    A.Flip(p=0.5),
    A.RandomResizedCrop(IMG_SIZE, IMG_SIZE, scale=(0.9,1.0), ratio=(0.95,1.05), p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
    A.Normalize(),
    ToTensorV2(),
])
val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])
# PyTorch Dataset wrapper for HuggingFace dataset
class HFDataset(Dataset):
    def __init__(self, hf_dataset, transforms=None):
        self.ds = hf_dataset
        self.transforms = transforms
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[int(idx)]
        img = np.array(item['image'])
        if self.transforms:
            img = self.transforms(image=img)['image']
        label = int(item['label'])
        return img, label
# Dataloaders
BATCH_SIZE = 32
train_dataset = HFDataset(ds_train, transforms=train_transforms)
val_dataset = HFDataset(ds_val, transforms=val_transforms)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print('Train samples:', len(train_dataset), 'Val samples:', len(val_dataset))

In [ ]:
# 2) Model: create timm model with custom head
import timm
import torch.nn as nn

MODEL_NAME = 'convnext_tiny'  # or 'efficientnet_b2'
NUM_CLASSES = 10

# Load pretrained model without classifier head
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, global_pool='avg')
feature_dim = model.num_features
print(f'Model: {MODEL_NAME}')
print(f'Feature dimension: {feature_dim}')

# Define custom head: Linear → BatchNorm → GELU → Dropout → Linear(10)
class CustomHead(nn.Module):
    def __init__(self, in_features, num_classes, dropout=0.4):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes)
        )
    
    def forward(self, x):
        return self.head(x)

# Attach custom head to model
model.head = CustomHead(feature_dim, NUM_CLASSES)
model = model.to(device)

print(f'\nModel architecture:')
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 2. Model Architecture

### Base Model: ConvNeXt-Tiny
ConvNeXt is a modern CNN architecture that incorporates design principles from Vision Transformers while maintaining the efficiency of convolutional networks. It's particularly well-suited for image classification tasks with limited training data due to strong pretrained representations.

**Key advantages:**
- Pretrained on ImageNet-1K (1.28M images, 1000 classes)
- Efficient architecture with ~28M parameters
- Better feature extraction than older architectures (ResNet, EfficientNet)
- Good balance between accuracy and inference speed

### Custom Classification Head
Replacing the standard linear classifier with a more sophisticated head:
- **Linear layer**: Projects features to same dimension for stability
- **BatchNorm1d**: Normalizes activations, helps training stability
- **GELU**: Smooth non-linearity (better than ReLU for fine-grained classification)
- **Dropout (0.4)**: Prevents overfitting on the relatively small galaxy dataset
- **Linear layer**: Final projection to 10 classes

This design provides better regularization and feature refinement than a single linear layer.

In [ ]:
# Load best model for evaluation
checkpoint = torch.load(save_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]} with Val F1: {checkpoint["best_val_f1"]:.4f}')

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
ax1.plot(training_history['train_loss'], label='Train Loss', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# F1 curve
ax2.plot(training_history['val_f1'], label='Val Macro-F1', marker='o', color='green')
ax2.axhline(y=best_val_f1, color='r', linestyle='--', label=f'Best F1: {best_val_f1:.4f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro-F1')
ax2.set_title('Validation Macro-F1')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Summary of all metrics
print('\n' + '=' * 70)
print('FINAL EVALUATION SUMMARY')
print('=' * 70)
print(f'Best Validation Macro-F1:     {best_val_f1:.4f}')
print(f'Test Macro-F1:                {macro_f1:.4f}')
print(f'Test Macro Precision:         {macro_precision:.4f}')
print(f'Test Macro Recall:            {macro_recall:.4f}')
print(f'Test Macro ROC-AUC:           {roc_auc_macro:.4f}')
print('=' * 70)
print('\nAll figures saved:')
print('  - confusion_matrix.png')
print('  - per_class_metrics.png')
print('  - roc_auc_scores.png')
print('  - per_class_metrics.csv')
print('  - best_model.pth')
print('=' * 70)

In [ ]:
# Create summary table of robustness results
robustness_summary = pd.DataFrame({
    'Corruption Type': corruption_types,
    'Macro-F1': macro_f1_scores,
    'F1 Drop': [0, clean_macro_f1 - blur_macro_f1, clean_macro_f1 - noise_macro_f1, clean_macro_f1 - jpeg_macro_f1],
    'Drop %': [0, 100*(clean_macro_f1 - blur_macro_f1)/clean_macro_f1, 
               100*(clean_macro_f1 - noise_macro_f1)/clean_macro_f1,
               100*(clean_macro_f1 - jpeg_macro_f1)/clean_macro_f1]
})

print('\n' + '='*70)
print('ROBUSTNESS SUMMARY')
print('='*70)
print(robustness_summary.to_string(index=False))
print('='*70)

# Save to CSV
robustness_summary.to_csv('robustness_summary.csv', index=False)
print('\nRobustness summary saved to: robustness_summary.csv')

### Robustness Analysis Interpretation

**Key insights from robustness testing:**

1. **Most vulnerable corruption** - The corruption type with the largest macro-F1 drop indicates which image degradation most affects the model's learned features.

2. **Most robust classes** - Classes with minimal F1 drops rely on features that are preserved under corruption (e.g., large-scale morphology, overall shape).

3. **Most vulnerable classes** - Classes with large drops depend on fine details that are corrupted (e.g., spiral arms, small-scale texture, color gradients).

4. **Relative performance** - If all classes drop similarly, the model uses global features. If drops vary widely, the model uses class-specific feature types (some fragile, some robust).

### Recommendations for Improvement
- **For blur sensitivity**: Add more blur augmentation during training, or use deconvolution preprocessing
- **For noise sensitivity**: Train with heavier noise augmentation, or use denoising autoencoders
- **For compression sensitivity**: Include JPEG compression in training augmentation pipeline
- **General**: Ensemble models trained on different augmentation strategies

In [ ]:
# Visualize robustness comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Macro-F1 comparison
corruption_types = ['Clean', 'Blur', 'Noise', 'JPEG']
macro_f1_scores = [clean_macro_f1, blur_macro_f1, noise_macro_f1, jpeg_macro_f1]
colors_bar = ['green', 'orange', 'coral', 'crimson']

axes[0].bar(corruption_types, macro_f1_scores, color=colors_bar, alpha=0.8)
axes[0].set_ylabel('Macro-F1', fontsize=11)
axes[0].set_title('Macro-F1 Under Different Corruptions', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 1.0])
axes[0].grid(axis='y', alpha=0.3)
for i, (typ, score) in enumerate(zip(corruption_types, macro_f1_scores)):
    axes[0].text(i, score + 0.02, f'{score:.3f}', ha='center', fontsize=10)

# Per-class F1 comparison (heatmap)
f1_comparison = np.array([clean_f1, blur_f1, noise_f1, jpeg_f1])
im = axes[1].imshow(f1_comparison, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_yticks(range(4))
axes[1].set_yticklabels(corruption_types)
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_xlabel('Class', fontsize=11)
axes[1].set_title('Per-Class F1-Score Heatmap', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='F1-Score')

plt.tight_layout()
plt.savefig('robustness_comparison.png', dpi=150, bbox_inches='tight')
print('\nRobustness comparison plot saved to: robustness_comparison.png')
plt.show()

In [ ]:
# Test 3: JPEG Compression
print('\n' + '='*70)
print('ROBUSTNESS TEST 3: JPEG COMPRESSION (quality=30)')
print('='*70)

jpeg_dataset = CorruptedDataset(ds_val, apply_jpeg_compression, val_transforms)
jpeg_loader = DataLoader(jpeg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

jpeg_preds, jpeg_probs, jpeg_targets = evaluate_model(model, jpeg_loader, device)
jpeg_precision, jpeg_recall, jpeg_f1, _ = precision_recall_fscore_support(
    jpeg_targets, jpeg_preds, average=None, zero_division=0
)
jpeg_macro_f1 = np.mean(jpeg_f1)

print(f'\nJPEG Macro-F1: {jpeg_macro_f1:.4f} (drop: {clean_macro_f1 - jpeg_macro_f1:.4f})')
print('\nPer-Class Performance:')
print(f'{"Class":<8} {"Clean F1":<12} {"JPEG F1":<12} {"Drop":<12}')
print('-' * 50)
for i in range(NUM_CLASSES):
    drop = clean_f1[i] - jpeg_f1[i]
    print(f'{i:<8} {clean_f1[i]:<12.4f} {jpeg_f1[i]:<12.4f} {drop:<12.4f}')

In [ ]:
# Test 2: Gaussian Noise
print('\n' + '='*70)
print('ROBUSTNESS TEST 2: GAUSSIAN NOISE (std=15)')
print('='*70)

noise_dataset = CorruptedDataset(ds_val, apply_gaussian_noise, val_transforms)
noise_loader = DataLoader(noise_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

noise_preds, noise_probs, noise_targets = evaluate_model(model, noise_loader, device)
noise_precision, noise_recall, noise_f1, _ = precision_recall_fscore_support(
    noise_targets, noise_preds, average=None, zero_division=0
)
noise_macro_f1 = np.mean(noise_f1)

print(f'\nNoisy Macro-F1: {noise_macro_f1:.4f} (drop: {clean_macro_f1 - noise_macro_f1:.4f})')
print('\nPer-Class Performance:')
print(f'{"Class":<8} {"Clean F1":<12} {"Noise F1":<12} {"Drop":<12}')
print('-' * 50)
for i in range(NUM_CLASSES):
    drop = clean_f1[i] - noise_f1[i]
    print(f'{i:<8} {clean_f1[i]:<12.4f} {noise_f1[i]:<12.4f} {drop:<12.4f}')

In [ ]:
# Test 1: Gaussian Blur
print('\n' + '='*70)
print('ROBUSTNESS TEST 1: GAUSSIAN BLUR (kernel_size=5)')
print('='*70)

blur_dataset = CorruptedDataset(ds_val, apply_gaussian_blur, val_transforms)
blur_loader = DataLoader(blur_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

blur_preds, blur_probs, blur_targets = evaluate_model(model, blur_loader, device)
blur_precision, blur_recall, blur_f1, _ = precision_recall_fscore_support(
    blur_targets, blur_preds, average=None, zero_division=0
)
blur_macro_f1 = np.mean(blur_f1)

print(f'\nBlurred Macro-F1: {blur_macro_f1:.4f} (drop: {clean_macro_f1 - blur_macro_f1:.4f})')
print('\nPer-Class Performance:')
print(f'{"Class":<8} {"Clean F1":<12} {"Blur F1":<12} {"Drop":<12}')
print('-' * 50)
for i in range(NUM_CLASSES):
    drop = clean_f1[i] - blur_f1[i]
    print(f'{i:<8} {clean_f1[i]:<12.4f} {blur_f1[i]:<12.4f} {drop:<12.4f}')

In [ ]:
# Store clean validation metrics for comparison
clean_preds, clean_probs, clean_targets = val_preds, val_probs, val_targets
clean_precision, clean_recall, clean_f1, _ = precision_recall_fscore_support(
    clean_targets, clean_preds, average=None, zero_division=0
)
clean_macro_f1 = np.mean(clean_f1)

print('Clean validation baseline:')
print(f'  Macro-F1: {clean_macro_f1:.4f}')
print(f'  Per-class F1: {clean_f1}')

In [ ]:
# 5) Robustness Testing: Evaluate under corruptions
import io
from PIL import Image as PILImage

# Corruption functions
def apply_gaussian_blur(image, kernel_size=5):
    """Apply Gaussian blur to image"""
    return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)

def apply_gaussian_noise(image, mean=0, std=15):
    """Apply Gaussian noise to image"""
    noise = np.random.normal(mean, std, image.shape).astype(np.float32)
    noisy = image.astype(np.float32) + noise
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy

def apply_jpeg_compression(image, quality=30):
    """Apply JPEG compression to image"""
    pil_img = PILImage.fromarray(image)
    buffer = io.BytesIO()
    pil_img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    compressed = PILImage.open(buffer)
    return np.array(compressed)

# Custom dataset wrapper for applying corruptions
class CorruptedDataset(Dataset):
    def __init__(self, hf_dataset, corruption_fn, transforms):
        self.ds = hf_dataset
        self.corruption_fn = corruption_fn
        self.transforms = transforms
    
    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, idx):
        item = self.ds[int(idx)]
        img = np.array(item['image'])
        # Apply corruption
        img = self.corruption_fn(img)
        # Apply standard transforms
        img = self.transforms(image=img)['image']
        label = int(item['label'])
        return img, label

print('Robustness testing functions defined')

## 5. Robustness Testing

### Motivation
Real-world astronomical observations face various degradations:
- **Atmospheric turbulence**: Causes blur (seeing conditions)
- **Telescope tracking errors**: Motion blur
- **Detector noise**: CCD/CMOS sensor noise
- **Data compression**: FITS/JPEG compression for storage and transmission
- **Light pollution**: Background noise

Testing robustness ensures the model generalizes to imperfect observational conditions.

### Corruption Types

#### 1. Gaussian Blur (kernel_size=5)
**Simulates:** Atmospheric turbulence ("seeing"), optical aberrations, focus issues

**Physical interpretation:**
- Models point spread function (PSF) broadening
- Reduces fine spatial details (spiral arms, star-forming regions)
- Common in ground-based telescopes (vs space-based Hubble)

**Expected impact:** 
- Galaxies with fine structures (spirals, irregular) show larger F1 drops
- Smooth ellipticals are more robust

#### 2. Gaussian Noise (std=15)
**Simulates:** Detector read noise, photon shot noise, dark current

**Physical interpretation:**
- Low signal-to-noise ratio (SNR) observations
- Faint galaxies at high redshift
- Short exposure times

**Expected impact:**
- Uniform degradation across all classes
- Boundary features become harder to detect
- May affect all classes similarly

#### 3. JPEG Compression (quality=30)
**Simulates:** Data storage/transmission compression artifacts

**Physical interpretation:**
- Block artifacts from DCT compression
- Loss of high-frequency details
- Realistic for archival data or citizen science projects

**Expected impact:**
- Edge artifacts may confuse structural classification
- Texture-based features degraded
- Classes with detailed structures affected most

### Performance Drops: What They Reveal
- **Small drops (< 5%)**: Model is robust to this corruption
- **Moderate drops (5-15%)**: Model relies partially on features affected by corruption
- **Large drops (> 15%)**: Model heavily depends on corrupted features
- **Class-specific patterns**: Reveals which morphological features each class relies on

In [ ]:
# ROC-AUC (One-vs-Rest multi-class)
try:
    # Convert targets to one-hot encoding
    y_onehot = np.zeros((len(val_targets), NUM_CLASSES))
    y_onehot[np.arange(len(val_targets)), val_targets] = 1
    
    # Compute macro and per-class ROC-AUC
    roc_auc_macro = roc_auc_score(y_onehot, val_probs, average='macro', multi_class='ovr')
    roc_auc_per_class = roc_auc_score(y_onehot, val_probs, average=None, multi_class='ovr')
    
    print('\nROC-AUC Scores (One-vs-Rest):')
    print('=' * 70)
    for i, auc in enumerate(roc_auc_per_class):
        print(f'  Class {i}: {auc:.4f}')
    print('=' * 70)
    print(f'  Macro-Average ROC-AUC: {roc_auc_macro:.4f}')
    
    # Visualize per-class ROC-AUC
    plt.figure(figsize=(10, 5))
    plt.bar(range(NUM_CLASSES), roc_auc_per_class, color='mediumvioletred', alpha=0.7)
    plt.axhline(y=roc_auc_macro, color='red', linestyle='--', linewidth=2, label=f'Macro: {roc_auc_macro:.3f}')
    plt.xlabel('Class', fontsize=11)
    plt.ylabel('ROC-AUC', fontsize=11)
    plt.title('Per-Class ROC-AUC (One-vs-Rest)', fontsize=12, fontweight='bold')
    plt.xticks(range(NUM_CLASSES))
    plt.ylim([0.5, 1.0])
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_auc_scores.png', dpi=150, bbox_inches='tight')
    print('\nROC-AUC plot saved to: roc_auc_scores.png')
    plt.show()
    
except Exception as e:
    print(f'Error computing ROC-AUC: {e}')
    roc_auc_macro = float('nan')

In [ ]:
# Visualize per-class metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Precision
axes[0].bar(range(NUM_CLASSES), precision, color='steelblue', alpha=0.7)
axes[0].axhline(y=macro_precision, color='red', linestyle='--', linewidth=2, label=f'Macro: {macro_precision:.3f}')
axes[0].set_xlabel('Class', fontsize=11)
axes[0].set_ylabel('Precision', fontsize=11)
axes[0].set_title('Per-Class Precision', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Recall
axes[1].bar(range(NUM_CLASSES), recall, color='seagreen', alpha=0.7)
axes[1].axhline(y=macro_recall, color='red', linestyle='--', linewidth=2, label=f'Macro: {macro_recall:.3f}')
axes[1].set_xlabel('Class', fontsize=11)
axes[1].set_ylabel('Recall', fontsize=11)
axes[1].set_title('Per-Class Recall', fontsize=12, fontweight='bold')
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# F1-Score
axes[2].bar(range(NUM_CLASSES), f1, color='darkorange', alpha=0.7)
axes[2].axhline(y=macro_f1, color='red', linestyle='--', linewidth=2, label=f'Macro: {macro_f1:.3f}')
axes[2].set_xlabel('Class', fontsize=11)
axes[2].set_ylabel('F1-Score', fontsize=11)
axes[2].set_title('Per-Class F1-Score', fontsize=12, fontweight='bold')
axes[2].set_xticks(range(NUM_CLASSES))
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_metrics.png', dpi=150, bbox_inches='tight')
print('Per-class metrics plot saved to: per_class_metrics.png')
plt.show()

In [ ]:
# Per-class metrics: Precision, Recall, F1
precision, recall, f1, support = precision_recall_fscore_support(
    val_targets, val_preds, average=None, zero_division=0
)

# Create per-class metrics table
import pandas as pd

metrics_df = pd.DataFrame({
    'Class': range(NUM_CLASSES),
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

print('\nPer-Class Metrics:')
print('=' * 70)
print(metrics_df.to_string(index=False))
print('=' * 70)

# Compute macro averages
macro_precision = np.mean(precision)
macro_recall = np.mean(recall)
macro_f1 = np.mean(f1)

print(f'\nMacro-Averaged Metrics:')
print(f'  Precision: {macro_precision:.4f}')
print(f'  Recall:    {macro_recall:.4f}')
print(f'  F1-Score:  {macro_f1:.4f}')

# Save metrics to CSV
metrics_df.to_csv('per_class_metrics.csv', index=False)
print(f'\nPer-class metrics saved to: per_class_metrics.csv')

In [ ]:
# 4) Evaluation: Confusion matrix and per-class metrics
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import seaborn as sns

# Get predictions on validation set
val_preds, val_probs, val_targets = evaluate_model(model, val_loader, device)

# Compute confusion matrix
cm = confusion_matrix(val_targets, val_preds)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix - Galaxy Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
print('Confusion matrix saved to: confusion_matrix.png')
plt.show()

print(f'\nConfusion Matrix:\n{cm}')

## 4. Model Evaluation

### Metrics Overview
**Confusion Matrix** - Visualizes which galaxy types are confused with each other. Off-diagonal elements indicate systematic misclassifications.

**Per-Class Metrics:**
- **Precision**: Of all galaxies predicted as class X, what fraction are actually class X? (Important when false positives are costly)
- **Recall**: Of all true class X galaxies, what fraction did we correctly identify? (Important when false negatives are costly)
- **F1-Score**: Harmonic mean of precision and recall, balanced measure of performance

**Macro-F1**: Average F1 across all classes (treats each class equally, regardless of sample size)

**ROC-AUC (One-vs-Rest)**: Measures how well the model separates each class from all others. Values close to 1.0 indicate excellent discrimination.

### Expected Patterns
- **Spiral galaxies** (if present) might be confused with barred spirals due to similar structures
- **Elliptical galaxies** are typically easier to classify due to distinct smooth morphology
- **Irregular galaxies** may show lower performance due to high intra-class variation
- **Edge-on galaxies** might be challenging due to limited visible features

In [ ]:
# Training loop with mixed precision and early stopping
from tqdm.auto import tqdm

training_history = {
    'train_loss': [],
    'val_f1': [],
    'lr': []
}

print('Starting training...\n')

for epoch in range(1, EPOCHS + 1):
    # Training phase
    model.train()
    epoch_loss = 0.0
    num_samples = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        batch_size = images.size(0)
        epoch_loss += loss.item() * batch_size
        num_samples += batch_size
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Calculate average training loss
    avg_train_loss = epoch_loss / num_samples
    
    # Validation phase
    val_preds, val_probs, val_targets = evaluate_model(model, val_loader, device)
    val_macro_f1 = compute_macro_f1(val_preds, val_targets)
    
    # Update scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    training_history['train_loss'].append(avg_train_loss)
    training_history['val_f1'].append(val_macro_f1)
    training_history['lr'].append(current_lr)
    
    # Print epoch results
    print(f'Epoch {epoch}/{EPOCHS}:')
    print(f'  Train Loss: {avg_train_loss:.4f}')
    print(f'  Val Macro-F1: {val_macro_f1:.4f}')
    print(f'  Learning Rate: {current_lr:.6f}')
    
    # Early stopping check
    if val_macro_f1 > best_val_f1 + 1e-5:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': best_val_f1,
            'training_history': training_history
        }, save_path)
        
        print(f'  ✓ Best model saved (Val F1: {best_val_f1:.4f})')
    else:
        patience_counter += 1
        print(f'  No improvement ({patience_counter}/{patience})')
        
        if patience_counter >= patience:
            print(f'\nEarly stopping triggered at epoch {epoch}')
            break
    
    print()

print(f'\nTraining completed!')
print(f'Best validation macro-F1: {best_val_f1:.4f}')
print(f'Model saved to: {save_path}')

In [ ]:
# Helper functions for evaluation
def evaluate_model(model, loader, device):
    """Evaluate model and return predictions, probabilities, and targets"""
    model.eval()
    all_preds = []
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            
            with autocast():
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
            
            all_preds.append(outputs.argmax(dim=1).cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
    
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)
    targets = np.concatenate(all_targets)
    
    return preds, probs, targets

def compute_macro_f1(preds, targets):
    """Compute macro-averaged F1 score"""
    _, _, f1_scores, _ = precision_recall_fscore_support(
        targets, preds, average=None, zero_division=0
    )
    macro_f1 = float(np.mean(f1_scores))
    return macro_f1

print('Evaluation functions defined')

In [ ]:
# 3) Training setup: loss, optimizer, scheduler, mixed precision
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Mixed precision scaler
scaler = GradScaler()

# Early stopping parameters
best_val_f1 = -1.0
patience = 5
patience_counter = 0
save_path = 'best_model.pth'

print('Training configuration:')
print(f'  Optimizer: AdamW (lr=1e-4, weight_decay=1e-4)')
print(f'  Loss: Weighted CrossEntropyLoss')
print(f'  Scheduler: CosineAnnealingLR')
print(f'  Mixed precision: Enabled')
print(f'  Early stopping: patience={patience}, metric=validation macro-F1')
print(f'  Max epochs: {EPOCHS}')

## 3. Training Strategy

### Loss Function
**Weighted Cross-Entropy Loss** - Uses computed class weights to handle imbalanced data. Classes with fewer samples receive higher loss weights, forcing the model to pay more attention to minority classes.

### Optimization
- **AdamW optimizer**: Adam with decoupled weight decay (better generalization)
- **Learning rate**: 1e-4 (conservative for fine-tuning pretrained models)
- **Weight decay**: 1e-4 (L2 regularization to prevent overfitting)

### Learning Rate Scheduling
**Cosine Annealing** - Gradually reduces learning rate following a cosine curve. This allows:
- Fast initial learning with higher LR
- Fine-grained refinement as LR approaches zero
- Smooth convergence without sudden drops

### Mixed Precision Training
Uses PyTorch's Automatic Mixed Precision (AMP) with `autocast` and `GradScaler`:
- **Faster training**: FP16 operations are 2-3× faster on modern GPUs
- **Lower memory**: Allows larger batch sizes
- **Maintained accuracy**: Critical operations still use FP32

### Early Stopping
Monitors **validation macro-F1** (not loss or accuracy) because:
- **Macro-F1** treats all classes equally, crucial for imbalanced datasets
- Prevents overfitting by stopping when validation performance plateaus
- Patience=5 epochs allows for noise in validation metrics

## Pipeline Summary

### Complete Implementation
✅ **Data Pipeline** - HuggingFace dataset with class-balanced sampling and augmentation  
✅ **Model** - ConvNeXt-Tiny pretrained backbone with custom 10-class head  
✅ **Training** - Mixed precision, weighted loss, cosine LR, early stopping on macro-F1  
✅ **Evaluation** - Confusion matrix, per-class precision/recall/F1, ROC-AUC  
✅ **Robustness** - Performance analysis under blur, noise, and compression  

### Saved Artifacts
📁 **Model & Metrics:**
- `best_model.pth` - Best model checkpoint (can be loaded for inference)
- `per_class_metrics.csv` - Detailed classification metrics
- `robustness_summary.csv` - Robustness test results

📊 **Visualizations:**
- `confusion_matrix.png` - Confusion matrix heatmap
- `per_class_metrics.png` - Precision/Recall/F1 bar charts
- `roc_auc_scores.png` - ROC-AUC by class
- `robustness_comparison.png` - Macro-F1 and per-class F1 under corruptions

### Key Features
🔬 **Reproducible** - Fixed random seeds, deterministic algorithms  
⚡ **GPU-Optimized** - Mixed precision training, efficient dataloaders  
📈 **Research-Grade** - Publication-ready metrics and visualizations  
🛡️ **Robust Testing** - Real-world corruption scenarios  

### Usage
Run all cells sequentially from top to bottom. Training takes ~10-30 minutes on a modern GPU (depending on dataset size). The pipeline automatically saves all results and can be interrupted/resumed by loading the checkpoint.